# All imports

In [39]:
import requests
import os
import json
from dotenv import load_dotenv
import time
from datetime import datetime

# Load the keys from the .env file which contain keys information into the notebook

In [8]:
load_dotenv("../../.env")    # the notebook runs in notebooks/, so go up to the top-level folder to get .env file

app_id = os.environ["ADZUNA_APP_ID"]
app_key = os.environ["ADZUNA_APP_KEY"]
print("keys loaded")

keys loaded


# The request 
Building the request

In [40]:
url = "https://api.adzuna.com/v1/api/jobs/ca/search/1"
params = {
    "app_id": app_id,
    "app_key": app_key,
    "what": "data analyst",
    "where": "Calgary",
    "results_per_page": 100,
}

print(url)
print(list(params.keys()))

https://api.adzuna.com/v1/api/jobs/ca/search/1
['app_id', 'app_key', 'what', 'where', 'results_per_page']


Send one request and checks status

In [9]:
response = requests.get(url, params=params)
print(response.status_code)

if response.status_code == 200:
    print(response.json()["count"], "matching jobs")

200
42 matching jobs


Saving the raw response to the raw data file

In [10]:
import json 

if response.status_code == 200:
    with open("../data/raw/adzuna_ca_calgary_data_analyst.json", "w") as f:
        json.dump(response.json(),f)
    print("saved")

saved


Adding log line without re running the request to save API calls

In [13]:
from datetime import datetime

print(datetime.now(), "Obtained Key | Adzuna | Set Params for Data Analyst in Calgary | Sent Request | Status", response.status_code, "| saved to data/raw/")

2026-10-15 23:03:02.930846 Obtained Key | Adzuna | Set Params for Data Analyst in Calgary | Sent Request | Status 200 | saved to data/raw/


Loops each roles and cities and makes a file for each pair, sends it to raw data file, logs results

In [41]:
roles = ["data analyst", "data engineer", "data scientist"]
cities = ["Calgary", "Toronto", "Vancouver", "Montreal", "Ottawa"]

for city in cities:
    for role in roles:
        filename = f"../data/raw/adzuna_ca_{city.lower()}_{role.replace(' ', '_')}.json"

        if os.path.exists(filename):
            print("skip, already saved:",filename)
            continue

        params = {
            "app_id": app_id,
            "app_key": app_key,
            "what": role,
            "where": city,
            "results_per_page": 100,
        }
        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()
            with open(filename, 'w') as f:
                json.dump(data, f)
            print(datetime.now(), "|", role, "|", city, "| status 200 |", data["count"], "jobs |", len(data["results"]), "saved")
        else:
            print(datetime.now(), "|", role, "|", city, "| status", response.status_code, "| NOT saved")

        time.sleep(1)


skip, already saved: ../data/raw/adzuna_ca_calgary_data_analyst.json
2026-10-16 00:23:32.805274 | data engineer | Calgary | status 200 | 249 jobs | 50 saved
2026-10-16 00:23:34.877072 | data scientist | Calgary | status 200 | 131 jobs | 50 saved
2026-10-16 00:23:36.665280 | data analyst | Toronto | status 200 | 317 jobs | 50 saved
2026-10-16 00:23:38.386712 | data engineer | Toronto | status 200 | 1624 jobs | 50 saved
2026-10-16 00:23:40.162835 | data scientist | Toronto | status 200 | 1189 jobs | 50 saved
2026-10-16 00:23:41.926205 | data analyst | Vancouver | status 200 | 22 jobs | 22 saved
2026-10-16 00:23:43.809842 | data engineer | Vancouver | status 200 | 126 jobs | 50 saved
2026-10-16 00:23:45.705201 | data scientist | Vancouver | status 200 | 92 jobs | 50 saved
2026-10-16 00:23:47.589503 | data analyst | Montreal | status 200 | 105 jobs | 50 saved
2026-10-16 00:23:49.358642 | data engineer | Montreal | status 200 | 308 jobs | 50 saved
2026-10-16 00:23:51.115572 | data scientist

In [42]:
import math

extra_calls = 0

for name in sorted(os.listdir("../data/raw")):
    if name.startswith("adzuna_ca_") and name.endswith(".json"):
        with open(f"../data/raw/{name}") as f:
            data = json.load(f)
        total = data["count"]
        pages = math.ceil(total / 50)
        extra_calls += pages - 1
        print(name, "|", total, "jobs |", pages, "page(s)")

print("extra calls needed for everything:", extra_calls)


adzuna_ca_calgary_data_analyst.json | 42 jobs | 1 page(s)
adzuna_ca_calgary_data_engineer.json | 249 jobs | 5 page(s)
adzuna_ca_calgary_data_scientist.json | 131 jobs | 3 page(s)
adzuna_ca_montreal_data_analyst.json | 105 jobs | 3 page(s)
adzuna_ca_montreal_data_engineer.json | 308 jobs | 7 page(s)
adzuna_ca_montreal_data_scientist.json | 239 jobs | 5 page(s)
adzuna_ca_ottawa_data_analyst.json | 22 jobs | 1 page(s)
adzuna_ca_ottawa_data_engineer.json | 114 jobs | 3 page(s)
adzuna_ca_ottawa_data_scientist.json | 74 jobs | 2 page(s)
adzuna_ca_toronto_data_analyst.json | 317 jobs | 7 page(s)
adzuna_ca_toronto_data_engineer.json | 1624 jobs | 33 page(s)
adzuna_ca_toronto_data_scientist.json | 1189 jobs | 24 page(s)
adzuna_ca_vancouver_data_analyst.json | 22 jobs | 1 page(s)
adzuna_ca_vancouver_data_engineer.json | 126 jobs | 3 page(s)
adzuna_ca_vancouver_data_scientist.json | 92 jobs | 2 page(s)
extra calls needed for everything: 85
